# **Lab 01 - Introduction to Python, Gymnasium and Formulating RL Problem**

##### Copyright by UIT-NC@NT549

## **Some instructions before getting started**:
<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">

Start the Kernel: At the top right, choose <strong>Select Kernel ➞ Python Environments...</strong>

You can run all code blocks to check: From the menu bar, choose <strong>Run All</strong>.

Complete all code blocks marked with the comment <span style="font-family: monospace; font-weight: bold; color:white; background-color: green;"> ### YOU NEED TO WRITE YOUR CODE BELOW ### </span>
</div>

## Part 1: Getting familiar with Gymnasium

## Part 2: Custom Environment "VacuumCleaner"

## Part 3: Load Balancing Problem

In [ ]:
"""
Load Balancing Environment Simulation

This program simulates a load balancing environment where tasks are distributed among multiple servers. 
Each server has a queue with a limited size, and tasks are processed based on their arrival and the server's availability.

Key Components:
1. Task: Represents a task with a specific processing time.
2. Server: Represents a server that processes tasks. Each server has a queue to hold tasks waiting for processing.
3. LoadBalancingEnv: A custom OpenAI Gym environment that simulates the load balancing scenario. 
     The environment allows agents to decide which server to send a task to, aiming to maximize rewards by minimizing 
     dropped tasks and reducing waiting times.

The goal is to design a load balancing strategy that optimally distributes tasks among servers to maximize efficiency.

Classes:
- Task: Represents a task with a specific processing time.
- Server: Represents a server with a queue of limited size.
- LoadBalancingEnv: Custom Gym environment for the load balancing problem.

Usage:
- The environment can be used to train reinforcement learning agents to learn optimal load balancing strategies.
"""

import random
from collections import deque

class Task:
      """A task with a unique id and required processing time (in time steps)."""

      def __init__(self, task_id: int, processing_time: int):
            self.task_id = task_id  # Unique identifier for tracking
            self.processing_time = processing_time  # Number of steps needed to finish


class Server:
      """
      A server with:
      - one currently running task
      - a waiting queue with finite capacity
      """

      def __init__(self, queue_capacity: int):
            self.queue_capacity = queue_capacity  # Max number of tasks waiting in queue
            self.queue = deque()  # FIFO queue of waiting tasks
            self.current_task = None  # Task being processed right now
            self.remaining_time = 0  # Steps left for current_task

      def run_one_step(self):
            """
            Execute one simulation step on this server.

            Returns:
                  completed_task (Task | None): task completed at this step, if any.
            """
            completed_task = None

            # Process one time unit for currently running task
            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            # HERE
            if self.current_task is not None:
                  self.remaining_time -= 1
                  if self.remaining_time == 0:
                        completed_task = self.current_task
                        self.current_task = None
                        self.remaining_time = 0

            # If server becomes idle, immediately pull next task from queue
            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            # HERE
            else:
                  if len(self.queue) > 0:
                        self.current_task = self.queue.popleft()
                        self.remaining_time = self.current_task.processing_time

            return completed_task

      def add_task(self, task: Task) -> bool:
            """
            Try to add a task to this server.

            Rules:
            - If server is idle: start processing immediately.
            - Else if queue has room: enqueue task.
            - Else: reject (drop) task.

            Returns:
                  bool: True if accepted, False if dropped.
            """
            # Start immediately if server is free
            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            # HERE
            if self.current_task is None: 
                  self.current_task = task
                  self.remaining_time = self.current_task.processing_time
                  return True

            # Otherwise enqueue if capacity allows
            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            # HERE
            else:
                  # Queue not full -> enqueue task
                  if len(self.queue) < self.queue_capacity:
                        self.queue.append(task)
                        return True

                  # Queue full -> task dropped
                  return False

      def queue_length(self) -> int:
            # Current number of waiting tasks (excluding running task)
            return len(self.queue)

**Test server**

In [ ]:
# Init server
server = Server(queue_capacity=3)

# Add 3 tasks into waiting queue
server.add_task(Task(1,2))
server.add_task(Task(2,3))
server.add_task(Task(3,3))

# Check state
print("Current task:", server.current_task.task_id)
print("Remaining time:", server.remaining_time)
print("Queue:", [task.task_id for task in server.queue])


In [ ]:
import gymnasium as gym
class LoadBalancingEnv(gym.Env):
      """
      Custom Gym environment for load balancing.

      Action:
            Choose a server index to receive the new incoming task.

      State:
            - each server's remaining processing time and queue length
            - global time step

      Metrics tracked:
            - total_created_tasks
            - accepted_tasks
            - dropped_tasks
            - completed_tasks
            - drop_rate
      """

      metadata = {"render.modes": ["human"]}

      def __init__(self, n_servers: int = 3, queue_capacity: int = 2, seed: int = None):
            super().__init__()
            self.n_servers = n_servers
            self.queue_capacity = queue_capacity
            self.rng = random.Random(seed)  # Local random generator for reproducibility

            # Create server pool
            self.servers = [Server(queue_capacity) for _ in range(n_servers)]
            self.time = 0  # Global simulation step
            self.total_reward = 0.0  # Accumulated reward over episode

            # Task tracking
            self.next_task_id = 0
            self.tasks_created = {}     # task_id -> Task
            self.tasks_completed = set()  # IDs of completed tasks
            self.tasks_dropped = set()  # IDs of dropped tasks
            self.tasks_accepted = set()  # IDs of accepted tasks

            # RL spaces
            # Define action space with n_servers 
            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            # HERE
            self.action_space = gym.spaces.Discrete(self.n_servers)

            # Define observation space as a dict containing server states and global time
            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            # HERE
            self.observation_space = gym.spaces.Dict({
                  "servers": gym.spaces.Tuple([
                        gym.spaces.Dict({
                              "remaining_time": gym.spaces.Box(low=0, high=float('inf'), shape=(), dtype=float),
                              "queue_length": gym.spaces.Discrete(self.queue_capacity + 1)
                        }) for _ in range(self.n_servers)
                  ]),
                  "time": gym.spaces.Box(low=0, high=float('inf'), shape=(), dtype=float)
            })


      def _new_task(self) -> Task:
            """Create one new incoming task with random processing time [1, 5]."""
            # Create a new task with a unique ID and random processing time between 1 and 5
            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            # HERE
            t = Task(self.next_task_id, self.rng.randint(1, 5))
            self.tasks_created[t.task_id] = t
            self.next_task_id += 1
            return t

      def reset(self, *, seed=None, options=None):
            """Reset environment state and all tracking metrics."""
            # Optional reseed to make episode deterministic from this point
            if seed is not None:
                  self.rng.seed(seed)

            # Reinitialize server states, time, and rewards
            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            # HERE
            self.servers = [Server(self.queue_capacity) for _ in range(self.n_servers)]
            self.time = 0
            self.total_reward = 0.0

            # Clear all task statistics
            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            # HERE
            self.next_task_id = 0
            self.tasks_created = {}
            self.tasks_completed = set()
            self.tasks_dropped = set()
            self.tasks_accepted = set()


            # Gymnasium-style reset return: (observation, info)
            return self._get_observation(), {}

      def step(self, action: int):
            """
            Run one simulation step:
            1) Advance all servers by one time step.
            2) Generate one new task.
            3) Route task to selected server.
            4) Compute reward and return transition tuple.
            """
            reward = 0.0

            # 1) Process running tasks on each server
            # For each server, call run_one_step() to advance processing. If a task completes, add to completed set and give a positive reward (+2.0)
            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            # HERE
            for server in self.servers:
                  completed_task = server.run_one_step()
                  if completed_task is not None:
                        self.tasks_completed.add(completed_task.task_id)
                        reward += 2.0


            # 2) Generate one new incoming task
            new_task = self._new_task()

            # 3) Route task based on action (selected server index)
            # Try to add the new task to the selected server. 
            # If accepted, add to accepted set and give a small reward (+0.5). 
            # If dropped, add to dropped set and give a strong penalty (-5.0).
            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            # HERE
            if(self.servers[action].add_task(new_task)):
                  self.tasks_accepted.add(new_task.task_id)
                  reward += 0.5
            else:
                  self.tasks_dropped.add(new_task.task_id)
                  reward -= 5
      
            # 4) Add congestion penalty proportional to queue sizes
            # To encourage the agent to balance load and avoid long 
            # queues, subtract a small penalty (e.g., -0.5) for each task waiting in any server's queue.
            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            # HERE
            for server in self.servers:
                  reward -= 0.5 * len(server.queue)

            # Update global counters
            self.total_reward += reward
            self.time += 1

            # This environment currently never ends by itself
            terminated = False
            truncated = False
            info = self._get_info()

            return self._get_observation(), reward, terminated, truncated, info

      def _get_observation(self):
            """Build observation dict from current system state."""
            # As a hint to design your observation, we provide the following structure:
            return {
                  "servers": tuple(
                         {
                               "remaining_time": float(server.remaining_time),
                               "queue_length": server.queue_length(),
                         }
                         for server in self.servers
                  ),
                  "time": float(self.time),
            }

      def _get_info(self):
            """Return useful metrics for logging/evaluation."""
            # Compute metrics based on requirements on your lab assignment. Here are some examples:
            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            # HERE
            total_created = len(self.tasks_created)
            dropped = len(self.tasks_dropped)
            completed = len(self.tasks_completed)
            accepted = len(self.tasks_accepted)

            # Safe division to avoid divide-by-zero at the beginning
            # Calculate some key performance metrics
            ### YOU NEED TO WRITE YOUR CODE BELOW ###
            # HERE
            if total_created > 0:
                  drop_rate = dropped / total_created
            else:
                  drop_rate = 0.0
            #...

            return {
                  # Some example info fields: time, dropped tasks, dop_rate, etc.
                  "time": self.time,
                  "total_created tasks": total_created,
                  "dropped tasks": dropped,
                  "completed tasks": completed,
                  "accepted tasks": accepted,
                  "drop rate": drop_rate
            }

In [ ]:
def load_balancing_policy(option="random", env=None):
      """Simple baseline policy."""
      if option == "random":
            # Uniform random server selection
            return env.action_space.sample()
      # Define other policies based on the option string
      ### YOU NEED TO WRITE YOUR CODE BELOW ###
      # HERE      

      # Round Robin
      elif option == "round_robin":
            
            # Declared rr_index if not exist
            if "rr_index" not in env.__dict__:
                  env.rr_index = 0
            
            server = env.rr_index
            
            env.rr_index += 1
            if env.rr_index >= env.n_servers:
                  env.rr_index = 0
            return server
      
      # Greedy
      elif option == "greedy":

            best_server = 0
            min_queue = len(env.servers[0].queue)

            index = 0
            for server in env.servers:
                  q = len(server.queue)
                  if q < min_queue:
                        min_queue = q
                        best_server = index

                  index += 1
            
            return best_server

      # Epsilon-Greedy Heuristic 
      elif option == "epsilon_greedy":

      # Declared epsilon = 0.1 ( 10% random server and 90% greedy server)      
            epsilon = 0.1

            if random.random() < epsilon:
                  return load_balancing_policy("random", env)
            else:
                  return load_balancing_policy("greedy", env)


      raise ValueError(f"Unsupported policy option: {option}")

In [ ]:
if __name__ == "__main__":

      episode_rewards = []
      log_lines = []
      option = input("Enter the argorithm: random / round_robin / greedy / epsilon_greedy")
      for episode in range(100):

            # Create environment and reset initial state
            env = LoadBalancingEnv(n_servers=3, queue_capacity=2, seed=None)
            obs, info = env.reset()

            # Run fixed number of simulation steps with random policy
            for step in range(20): # 20 episodes = 20 steps = 20 seconds of simulated time = 20 tasks created
                  action = load_balancing_policy(option, env=env)
                  obs, reward, terminated, truncated, info = env.step(action)

                  # Print every value you want to track at each step
                  # Save metrics to a csv file for later analysis
                  ### YOU NEED TO WRITE YOUR CODE BELOW ###
                  # HERE
                  step_header = f"\nEpisode {episode + 1} - Step {step + 1}"
                  print(step_header)
                  log_lines.append(step_header)

                  for i, server in enumerate(env.servers):
                        line = (
                              f"Server {i}: "
                              f"current_task={server.current_task.task_id if server.current_task else None}, "
                              f"remaining_time={server.remaining_time}, "
                              f"queue={[task.task_id for task in server.queue]}"
                        )
                        print(line)
                        log_lines.append(line)

            episode_rewards.append(env.total_reward)
            episode_line = f"Episode {episode + 1} reward: {env.total_reward:.2f}"
            print(episode_line)
            log_lines.append(episode_line)



      # Print summary statistics after simulation ends.
      ### YOU NEED TO WRITE YOUR CODE BELOW ###
      # HERE
      log_lines.append("\n=== Simulation Summary ===")
      log_lines.append(f"Average reward: {sum(episode_rewards)/len(episode_rewards):.2f}")
      save_metrics(log_lines)

      print("\n=== Simulation Summary ===")
      print(f"Average reward: {sum(episode_rewards)/len(episode_rewards):.2f}")




In [ ]:
import matplotlib.pyplot as plt

plt.plot(episode_rewards)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("Reward per Episode")
plt.show()

In [ ]:
file_name = input("Enter your file name: ")
file_path = input("Enter your file path: ")

def save_metrics(lines, file_name=file_name, file_path=file_path):
     full_path = f"{file_path}/{file_name}"

     with open(full_path, "w", encoding="utf-8") as f:
          f.write("\n".join(lines))

     print(f"Saved to {full_path}")


Evaluation and Analysis

## CONGRATULATIONS TEAM!

Congratulations to the team for completing Part 2,3 of Lab01 - Introduction to Python, Gymnasium and Formulating RL Problem.
Keep up the effort in the next sections.

References: https://gymnasium.farama.org/ 

Suggested additional practice resources: https://gymnasium.farama.org/introduction/create_custom_env/

## ADDITIONAL INFORMATION

**Author**: M.Sc. Phan Trung Phat - Department of Computer Networks and Communications, UIT

**Contact**: phatpt@uit.edu.vn
